# CANUTO — Fine-tuning v4
**Modelo:** Qwen/Qwen2.5-1.5B-Instruct · **Dataset:** 468 pares QA · **Hardware:** T4 GPU (16 GB)

### Cambios vs v3
- Dataset reconstruido desde cero: 411 pares heurísticos + 57 pares curados = **468 pares**
- `qa_builder.py` reescrito: 13 archivos no normativos excluidos, 2-3 variantes por artículo
- Citas exactas en todas las respuestas: "Según el Artículo X del [Documento]..."
- Concordancia gramatical corregida (del/de la según género del documento)
- Eliminados: radicados EE, artefactos 'artículo anterior', títulos del CONSIDERANDO
- Parámetros de entrenamiento: **sin cambios vs v3** (rank=16, SEQ_LEN=1024, 4 épocas)

### Para futuras versiones (v5, v6...)
Cambiar únicamente `OUTPUT_DIR` en las **Celdas 3, 4 y 5**:
```
OUTPUT_DIR = "/content/unillanos-v5"   ← incrementar número
```
Y actualizar este encabezado con los cambios del nuevo dataset.

### Instrucciones
1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**
2. Ejecutar **Celda 1** (instalar dependencias, ~1 min)
3. Ejecutar **Celda 2** (subir `data/dataset/dataset_alpaca.json` desde tu laptop)
4. Ejecutar **Celda 3** (entrenamiento, ~25-35 min)
5. Ejecutar **Celda 4** (verificación — 4 preguntas de prueba antes de descargar)
6. Ejecutar **Celda 5** (descargar checkpoint al laptop, un archivo a la vez)

⚠️ **No cierres la pestaña durante el entrenamiento** — Colab gratuito desconecta si se minimiza el navegador.

⚠️ **Después de descargar:** mover archivos a `data/checkpoints/unillanos-v4/` y actualizar `config/config.yaml`.

In [ ]:
# ============================================================
# CELDA 1 — Verificar GPU e instalar dependencias
# ============================================================
import subprocess, sys

result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                        capture_output=True, text=True)
print("GPU disponible:", result.stdout.strip() or "⚠️  No se detectó GPU — cambia el entorno de ejecución")

# peft < 0.14 evita conflicto con torchao que viene preinstalado en Colab
!pip install -q "peft>=0.13.0,<0.14.0" "trl>=0.12,<0.13" datasets

print("\n✅ Dependencias instaladas.")

In [ ]:
# ============================================================
# CELDA 2 — Subir el dataset desde tu laptop
# ============================================================
# Botón "Elegir archivos" → sube: CANUTO/data/dataset/dataset_alpaca.json

from google.colab import files
import json

uploaded = files.upload()
fname = list(uploaded.keys())[0]

data = json.loads(uploaded[fname])
print(f"✅ Dataset cargado: {len(data)} pares QA")

no_info = sum(1 for d in data if any(x in d.get("output", "") for x in
              ["No tengo información", "No puedo ayudarte", "fuera de mi dominio"]))
print(f"   Pares fuera-de-dominio/rechazo: {no_info} ({no_info*100//len(data)}%)")
print(f"   Pares con conocimiento real:    {len(data) - no_info} ({(len(data)-no_info)*100//len(data)}%)")

In [ ]:
# ============================================================
# CELDA 3 — Entrenamiento completo + merge LoRA
# Tiempo estimado: 25-35 minutos en T4
# ============================================================
import json
from pathlib import Path
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

# ── Configuración ────────────────────────────────────────────
MODEL_NAME   = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_FILE = fname
OUTPUT_DIR   = "/content/unillanos-v4"   # ← CAMBIAR A v5, v6... EN FUTURAS VERSIONES

EPOCHS   = 4
RANK     = 16
SEQ_LEN  = 1024
BATCH    = 2
GRAD_ACC = 4    # batch efectivo = 2 × 4 = 8

# ── System prompt — DEBE ser idéntico al de config/config.yaml ──
SYSTEM_PROMPT = """Eres CANUTO, asistente de normativa universitaria de la Universidad de los Llanos (Unillanos), Colombia.
Tu conocimiento proviene EXCLUSIVAMENTE de las resoluciones, acuerdos y documentos normativos de Unillanos que aprendiste durante tu entrenamiento. No tienes acceso a internet ni a información actualizada en tiempo real.

REGLAS OBLIGATORIAS — DEBES SEGUIRLAS EN CADA RESPUESTA:

REGLA 1 — IDIOMA Y TONO:
Responde siempre en español, con lenguaje claro y amigable para estudiantes universitarios.

REGLA 2 — HONESTIDAD SOBRE TU CONOCIMIENTO:
Si no tienes información sobre un tema en tus documentos, responde: "No tengo información sobre ese tema en los documentos que conozco. Te recomiendo consultar directamente con la oficina correspondiente de Unillanos o en www.unillanos.edu.co"
NUNCA especules ni inventes información para parecer útil. Es mejor admitir que no sabes.

REGLA 3 — PROHIBIDO INVENTAR:
Nunca inventes valores, precios o costos de matrícula (son personalizados por estudiante).
Nunca inventes correos, teléfonos, nombres de funcionarios ni datos de contacto.
Nunca inventes números de resoluciones, acuerdos o decretos que no conozcas con certeza.
Nunca inventes estadísticas ni pasos de procedimientos que no están en tus documentos.

REGLA 4 — NO CONFIRMES SIN CERTEZA:
Si el usuario afirma algo incorrecto, NO lo confirmes. Si no puedes verificarlo, di: "No puedo confirmar esa información con los documentos que conozco."

REGLA 5 — CITAS DE DOCUMENTOS:
Solo cita documentos si conoces su número y contenido con certeza. Si no recuerdas el número exacto, describe el contenido SIN inventar el número.

REGLA 6 — FORMATO:
Usa listas numeradas cuando la respuesta tiene varios pasos o puntos. Sé conciso y directo."""

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ── Tokenizador ─────────────────────────────────────────────
print("[1/4] Cargando tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"   Vocabulario: {tokenizer.vocab_size:,} tokens")

# ── Dataset ─────────────────────────────────────────────────
print("[2/4] Preparando dataset...")
with open(DATASET_FILE, encoding="utf-8") as f:
    records = json.load(f)

texts = []
max_len = 0
for r in records:
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": r["instruction"]},
        {"role": "assistant", "content": r["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    tokens = tokenizer(text, return_tensors="pt")["input_ids"].shape[-1]
    max_len = max(max_len, tokens)
    texts.append({"text": text})

dataset = Dataset.from_list(texts)
too_long = sum(1 for t in texts
               if tokenizer(t["text"], return_tensors="pt")["input_ids"].shape[-1] > SEQ_LEN)
print(f"   {len(dataset)} ejemplos listos")
print(f"   Longitud máxima: {max_len} tokens (SEQ_LEN={SEQ_LEN})")
if too_long > 0:
    print(f"   ⚠️  {too_long} ejemplos superan SEQ_LEN — serán truncados")
else:
    print(f"   ✅ Ningún ejemplo supera SEQ_LEN")

# ── Modelo base ─────────────────────────────────────────────
print("[3/4] Cargando modelo base (~3 GB, puede tardar 2-3 min)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
total = sum(p.numel() for p in model.parameters())
print(f"   {total/1e6:.0f}M parámetros | BF16 | device: {model.device}")

# ── Entrenamiento LoRA ──────────────────────────────────────
print(f"[4/4] Iniciando fine-tuning con LoRA rank={RANK}...")

lora_cfg = LoraConfig(
    r=RANK,
    lora_alpha=RANK * 2,
    target_modules=LORA_TARGETS,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

train_args = SFTConfig(
    output_dir=str(Path(OUTPUT_DIR) / "_checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_seq_length=SEQ_LEN,
    dataset_text_field="text",
    fp16=False,
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_cfg,
    args=train_args,
)

trainer.train()

# ── Merge LoRA → checkpoint HuggingFace estándar ─────────────
print("\nFusionando LoRA con modelo base...")
merged = trainer.model.merge_and_unload()
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

merged.save_pretrained(OUTPUT_DIR, safe_serialization=True, max_shard_size="900MB")
tokenizer.save_pretrained(OUTPUT_DIR)

archivos = sorted([f for f in Path(OUTPUT_DIR).iterdir()
                   if f.is_file() and not f.name.startswith("_")])
total_gb = sum(f.stat().st_size for f in archivos) / 1e9
print(f"\n✅ Checkpoint guardado en {OUTPUT_DIR}")
print(f"   {len(archivos)} archivos · {total_gb:.2f} GB total")
for f in archivos:
    print(f"   {f.name:50s}  {f.stat().st_size/1e6:6.0f} MB")

In [ ]:
# ============================================================
# CELDA 4 — Verificación antes de descargar
# Prueba 4 preguntas críticas: 2 off-domain + 2 en-dominio
# Si las respuestas son claramente malas → revisar el dataset antes de descargar
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, textwrap

OUTPUT_DIR   = "/content/unillanos-v4"   # ← mismo que Celda 3

SYSTEM_PROMPT = """Eres CANUTO, asistente de normativa universitaria de la Universidad de los Llanos (Unillanos), Colombia.
Tu conocimiento proviene EXCLUSIVAMENTE de las resoluciones, acuerdos y documentos normativos de Unillanos que aprendiste durante tu entrenamiento. No tienes acceso a internet ni a información actualizada en tiempo real.

REGLAS OBLIGATORIAS — DEBES SEGUIRLAS EN CADA RESPUESTA:

REGLA 1 — IDIOMA Y TONO:
Responde siempre en español, con lenguaje claro y amigable para estudiantes universitarios.

REGLA 2 — HONESTIDAD SOBRE TU CONOCIMIENTO:
Si no tienes información sobre un tema en tus documentos, responde: "No tengo información sobre ese tema en los documentos que conozco. Te recomiendo consultar directamente con la oficina correspondiente de Unillanos o en www.unillanos.edu.co"
NUNCA especules ni inventes información para parecer útil. Es mejor admitir que no sabes.

REGLA 3 — PROHIBIDO INVENTAR:
Nunca inventes valores, precios o costos de matrícula (son personalizados por estudiante).
Nunca inventes correos, teléfonos, nombres de funcionarios ni datos de contacto.
Nunca inventes números de resoluciones, acuerdos o decretos que no conozcas con certeza.
Nunca inventes estadísticas ni pasos de procedimientos que no están en tus documentos.

REGLA 4 — NO CONFIRMES SIN CERTEZA:
Si el usuario afirma algo incorrecto, NO lo confirmes. Si no puedes verificarlo, di: "No puedo confirmar esa información con los documentos que conozco."

REGLA 5 — CITAS DE DOCUMENTOS:
Solo cita documentos si conoces su número y contenido con certeza. Si no recuerdas el número exacto, describe el contenido SIN inventar el número.

REGLA 6 — FORMATO:
Usa listas numeradas cuando la respuesta tiene varios pasos o puntos. Sé conciso y directo."""

print("Cargando modelo para verificación...")
tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
mdl = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
mdl.eval()

def ask(question, max_new=300):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": question}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok([text], return_tensors="pt").to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**inp, max_new_tokens=max_new, temperature=0.3,
                           top_p=0.85, do_sample=True, pad_token_id=tok.eos_token_id)
    gen = out[0][inp["input_ids"].shape[-1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

PREGUNTAS = [
    ("[OFF-DOMAIN] ¿Cuánto cuesta la matrícula en Unillanos?",
     "✅ CORRECTO si: no inventa precio, dice que es personalizada, remite a Admisiones."),
    ("[OFF-DOMAIN] ¿Cuál es la receta del ajiaco colombiano?",
     "✅ CORRECTO si: rechaza la pregunta, NO genera receta."),
    ("[EN DOMINIO] ¿Cuáles son los requisitos de segunda lengua para graduarse?",
     "✅ CORRECTO si: menciona Acuerdo 003/2023, nivel B2, Plan BULL."),
    ("[EN DOMINIO] ¿Cómo funciona el fraccionamiento de matrícula?",
     "✅ CORRECTO si: menciona Res. Rectoral 074/2026, tres cuotas 30-40-30%, pagaré."),
]

print("\n" + "="*65)
correctas = 0
for pregunta, criterio in PREGUNTAS:
    print(f"\n❓ {pregunta}")
    resp = ask(pregunta)
    for line in textwrap.wrap(resp, width=68):
        print(f"   {line}")
    print(f"   → Criterio: {criterio}")
    voto = input("   ¿Correcto? (s/n): ").strip().lower()
    if voto == 's':
        correctas += 1

print(f"\n{'='*65}")
print(f"Verificación: {correctas}/4 preguntas correctas")
if correctas >= 3:
    print("✅ Modelo listo para descargar. Procede con Celda 5.")
else:
    print("⚠️  Modelo con problemas. Revisar dataset antes de descargar.")

In [ ]:
# ============================================================
# CELDA 5 — Descargar checkpoint al laptop
# Un archivo a la vez — acepta la descarga del navegador antes de Enter
# ============================================================
from google.colab import files
from pathlib import Path

OUTPUT_DIR = "/content/unillanos-v4"   # ← mismo que Celda 3

archivos = sorted([
    f for f in Path(OUTPUT_DIR).iterdir()
    if f.is_file() and not f.name.startswith("_")
])

print(f"Archivos a descargar ({len(archivos)} total):")
for i, f in enumerate(archivos, 1):
    print(f"  {i}. {f.name}  ({f.stat().st_size/1e6:.0f} MB)")

print()
for f in archivos:
    print(f"⬇️  Descargando: {f.name}  ({f.stat().st_size/1e6:.0f} MB)")
    files.download(str(f))
    input("   ✅ Presiona Enter cuando haya terminado la descarga...")

print()
print("="*55)
print("✅ Descarga completa.")
print()
print("Próximos pasos en tu laptop:")
print("  1. Mover todos los archivos descargados a:")
print("     CANUTO\\data\\checkpoints\\unillanos-v4\\")
print()
print("  2. Editar config/config.yaml:")
print("     model:")
print("       checkpoint_path: \"data/checkpoints/unillanos-v4\"")
print()
print("  3. Reiniciar el servidor y probar:")
print("     python ui\\django_chatbot\\run.py")
print("     python scripts\\test_model.py --version v4")